<a href="https://colab.research.google.com/github/aarti-311/AI-ML-Labs/blob/main/StockMarketNewsAnalysis_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LAB 7 : STOCK MARKET NEWS DATA ANALYSIS

In [ ]:
#Importing Libraries
import pandas as pd
import numpy as np
import nltk
import re
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# Reading the CSV file using pandas
df= pd.read_csv('Stock News Dataset.csv', encoding= 'unicode_escape')
# Displaying the first 10 rows of the dataframe
df.head(2)

,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top16,Top17,Top18,Top19,Top20,Top21,Top22,Top23,Top24,Top25
0,2000-01-03,0,A 'hindrance to operations': extracts from the...,Scorecard,Hughes' instant hit buoys Blues,Jack gets his skates on at ice-cold Alex,Chaos as Maracana builds up for United,Depleted Leicester prevail as Elliott spoils E...,Hungry Spurs sense rich pickings,Gunners so wide of an easy target,...,Flintoff injury piles on woe for England,Hunters threaten Jospin with new battle of the...,Kohl's successor drawn into scandal,The difference between men and women,"Sara Denver, nurse turned solicitor",Diana's landmine crusade put Tories in a panic,Yeltsin's resignation caught opposition flat-f...,Russian roulette,Sold out,Recovering a title
1,2000-01-04,0,Scorecard,The best lake scene,Leader: German sleaze inquiry,"Cheerio, boyo",The main recommendations,Has Cubie killed fees?,Has Cubie killed fees?,Has Cubie killed fees?,...,On the critical list,The timing of their lives,Dear doctor,Irish court halts IRA man's extradition to Nor...,Burundi peace initiative fades after rebels re...,PE points the way forward to the ECB,Campaigners keep up pressure on Nazi war crime...,Jane Ratcliffe,Yet more things you wouldn't know without the ...,Millennium bug fails to bite


In [ ]:
#Remove any null values from the dataset
df.dropna(inplace=True)

In [ ]:
# Using applymap(str) and dictionary notation
data = pd.DataFrame({
    'text': df.iloc[:, 2:].applymap(str).apply(' '.join, axis=1),
    'Label': df['Label']
})
data.head(3)

,text,Label
0,A 'hindrance to operations': extracts from the...,0
1,Scorecard The best lake scene Leader: German s...,0
2,Coventry caught on counter by Flo United's riv...,0


In [ ]:
pattern = r'\[[0-9]]*\]|\(.*?\)]|\d+|\s+|[^\w\s]'
data['text'] = data['text'].str.replace(pattern, ' ').str.lower()
data

C:\Users\DELL\AppData\Local\Temp\ipykernel_5404\221702016.py:2: FutureWarning: The default value of regex will change from True to False in a future version.
  data['text'] = data['text'].str.replace(pattern, ' ').str.lower()


,text,Label
0,hindrance operations extracts leaked reports s...,0
1,scorecard best lake scene leader german sleaze...,0
2,coventry caught counter flo united rivals road...,0
3,pilgrim knows progress thatcher facing ban mci...,1
4,hitches horlocks beckham united survive breast...,1
...,...,...
4257,barclays rbs shares suspended trading tanking ...,0
4258,scientists australia want save great barrier r...,1
4259,explosion airport istanbul yemeni former presi...,1
4260,jamaica proposes marijuana dispensers tourists...,1


In [ ]:
stop_words = set(stopwords.words('english'))

data['text'] = data['text'].apply(lambda x: ' '.join([word for word in word_tokenize(x) if word.lower() not in stop_words]))
data.head(5)

,text,Label
0,hindrance operations extracts leaked reports s...,0
1,scorecard best lake scene leader german sleaze...,0
2,coventry caught counter flo united rivals road...,0
3,pilgrim knows progress thatcher facing ban mci...,1
4,hitches horlocks beckham united survive breast...,1


In [ ]:
def stemming(text):
    return ' '.join([PorterStemmer().stem(word) for word in word_tokenize(text)])

data['text'] = data['text'].apply(stemming)
data.head(5)

,text,Label
0,hindranc oper extract leak report scorecard hu...,0
1,scorecard best lake scene leader german sleaz ...,0
2,coventri caught counter flo unit rival road ri...,0
3,pilgrim know progress thatcher face ban mcilro...,1
4,hitch horlock beckham unit surviv breast cance...,1


In [ ]:
def lemmatization(text):
    return ' '.join([WordNetLemmatizer().lemmatize(word) for word in word_tokenize(text)])

data['text'] = data['text'].apply(lemmatization)
data.head(5)

,text,Label
0,hindranc oper extract leak report scorecard hu...,0
1,scorecard best lake scene leader german sleaz ...,0
2,coventri caught counter flo unit rival road ri...,0
3,pilgrim know progress thatcher face ban mcilro...,1
4,hitch horlock beckham unit surviv breast cance...,1


In [ ]:
x = data['text']
y = data['Label']
train_data, test_data, train_labels, test_labels = train_test_split(x, y, test_size=0.2, random_state=1)

In [ ]:
#BOW
cv = CountVectorizer(max_features=1500)
x_train_cv  = cv.fit_transform(train_data)
x_test_cv = cv.transform(test_data)

In [ ]:
#Random forest Classifier
rf= RandomForestClassifier(max_depth=2, random_state=0)
rf.fit(x_train_cv, train_labels)
# Predicting the labels of test and train data
y_pred_test_cv = rf.predict(x_test_cv)
y_pred_train_cv = rf.predict(x_train_cv)


In [ ]:
print(classification_report(test_labels, y_pred_test_cv))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       362
           1       0.55      1.00      0.71       439

    accuracy                           0.55       801
   macro avg       0.27      0.50      0.35       801
weighted avg       0.30      0.55      0.39       801



C:\Users\DELL\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\DELL\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\DELL\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
# Initialize a TF-IDF vectorizer with a maximum of 1500 features
tfidf = TfidfVectorizer(max_features=1500)

# Fit and transform the training data using the vectorizer
x_train_tfidf = tfidf.fit_transform(train_data)

# Transform the test data using the trained vectorizer
x_test_tfidf = tfidf.transform(test_data)


In [ ]:
rf= RandomForestClassifier(max_depth=2, random_state=0)
rf.fit(x_train_tfidf, train_labels)
# Predicting the labels of test and train data
y_pred_test_tfidf = rf.predict(x_test_tfidf)
y_pred_train_tfidf = rf.predict(x_train_tfidf)

In [ ]:
print(classification_report(test_labels, y_pred_test_tfidf))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       362
           1       0.55      1.00      0.71       439

    accuracy                           0.55       801
   macro avg       0.27      0.50      0.35       801
weighted avg       0.30      0.55      0.39       801



C:\Users\DELL\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\DELL\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\DELL\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [ ]:
#WORD 2 VEC
# Tokenize the text data
tokenized_data = [word_tokenize(text) for text in data['text']]

# Train the word2vec model
model = Word2Vec(tokenized_data, min_count=1)
vocabulary= model.wv.key_to_index

# save the word2vec model to a file
model.save("word2vec.model")

# Get the vector representation of a word
word_vector = model.wv.get_vector('jamaica')
word_vector

array([-0.09478228,  0.07240704,  0.02932151,  0.07341257,  0.09174745,
       -0.11429637, -0.1443165 ,  0.42987528,  0.0324351 , -0.22664315,
        0.06164613,  0.05601228, -0.03467776,  0.05186922,  0.12379576,
        0.00825153, -0.12402225, -0.25640914,  0.0284093 , -0.19565776,
       -0.14179039, -0.02166441,  0.08398488, -0.03748883, -0.04107573,
       -0.01767811, -0.01586662,  0.0556052 , -0.1300177 ,  0.14499514,
        0.06674679, -0.02341257,  0.12221517, -0.12667966,  0.0666967 ,
       -0.05976514, -0.11224954, -0.11281159, -0.04633889, -0.17892966,
       -0.04605338, -0.14146073,  0.02271093, -0.1084827 ,  0.24146019,
       -0.17443007,  0.15275948, -0.06340952, -0.02048522,  0.26652882,
       -0.13165827, -0.01159054, -0.13465127,  0.19439943,  0.02798659,
       -0.09575144,  0.06131134,  0.16299795, -0.09933458,  0.05168581,
       -0.03071147,  0.09206203,  0.06647124,  0.02789233, -0.3478483 ,
        0.01603829,  0.16722298, -0.02186906, -0.25887313,  0.12

In [ ]:
print(model.wv.similarity(w1="marijuana", w2="australia"))

0.8252202


In [ ]:
similar= model.wv.most_similar('istanbul')
similar

[('calai', 0.98751300573349),
 ('hebdo', 0.9851545691490173),
 ('vandal', 0.9844745993614197),
 ('brussel', 0.9842716455459595),
 ('belong', 0.9835546612739563),
 ('firebomb', 0.9833953976631165),
 ('surround', 0.9827654957771301),
 ('cairo', 0.9827095866203308),
 ('downtown', 0.9826874732971191),
 ('crowd', 0.9820940494537354)]